# Sidebar Link Checker

This notebook checks all sidebar links to ensure they are connected to a page in the `/templates` directory. It will create any missing pages and maintain a status list of all links.

## Import Required Libraries

Import libraries necessary for file operations and data processing.

In [ ]:
import os
import json
import re
from pathlib import Path
import datetime

# Set up logging
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

## Configuration

Define paths and configuration settings for the sidebar link checking process.

In [ ]:
# Define paths
BASE_DIR = Path('d:/Projects/impressioncore')
TEMPLATES_DIR = BASE_DIR / 'templates'
CONFIG_DIR = BASE_DIR / 'config'
SIDEBAR_CONFIG_PATH = CONFIG_DIR / 'sidebar_config.json'

# Create directories if they don't exist
TEMPLATES_DIR.mkdir(exist_ok=True)
CONFIG_DIR.mkdir(exist_ok=True)

# Template for new pages
PAGE_TEMPLATE = """{% extends "base.html" %}
{% block title %}{{title}}{% endblock %}

{% block content %}
<div class="container mt-5">
    <h1>{{title}}</h1>
    <p>This page was automatically created by the sidebar link checker.</p>
    <p>Please replace this content with the appropriate information for this section.</p>
</div>
{% endblock %}
"""

## Load Sidebar Links

Load the sidebar links from the configuration file. If the file doesn't exist, create a sample config.

In [ ]:
def load_sidebar_links():
    """
    Load sidebar links from configuration file or create a sample if it doesn't exist.
    
    Returns:
        dict: Dictionary containing sidebar link configuration
    """
    if not SIDEBAR_CONFIG_PATH.exists():
        # Create a sample sidebar configuration
        sample_config = {
            "sidebar_sections": [
                {
                    "title": "Main Navigation",
                    "links": [
                        {"name": "Dashboard", "url": "/dashboard", "icon": "fa-tachometer-alt"},
                        {"name": "Users", "url": "/users", "icon": "fa-users"},
                        {"name": "Settings", "url": "/settings", "icon": "fa-cog"}
                    ]
                },
                {
                    "title": "Analytics",
                    "links": [
                        {"name": "Reports", "url": "/reports", "icon": "fa-chart-bar"},
                        {"name": "Statistics", "url": "/statistics", "icon": "fa-chart-line"},
                        {"name": "Metrics", "url": "/metrics", "icon": "fa-chart-pie"}
                    ]
                }
            ]
        }
        
        # Create the directory if it doesn't exist
        SIDEBAR_CONFIG_PATH.parent.mkdir(exist_ok=True)
        
        # Write the sample config
        with open(SIDEBAR_CONFIG_PATH, 'w') as file:
            json.dump(sample_config, file, indent=2)
        
        logger.info(f"Created sample sidebar configuration at {SIDEBAR_CONFIG_PATH}")
        return sample_config
    
    # Load existing configuration
    try:
        with open(SIDEBAR_CONFIG_PATH, 'r') as file:
            sidebar_config = json.load(file)
        logger.info(f"Loaded sidebar configuration from {SIDEBAR_CONFIG_PATH}")
        return sidebar_config
    except Exception as e:
        logger.error(f"Error loading sidebar configuration: {str(e)}")
        return {"sidebar_sections": []}

# Load the sidebar links
sidebar_config = load_sidebar_links()

## Check for Missing Pages

Iterate through the sidebar links and check if the corresponding pages exist in the templates directory.

In [ ]:
def extract_template_path(url):
    """
    Convert a URL path to a template path.
    
    Args:
        url (str): URL path like "/dashboard"
        
    Returns:
        str: Template path like "dashboard.html"
    """
    # Remove leading slash if present
    clean_url = url.lstrip('/')
    
    # Handle root URL
    if clean_url == "":
        return "index.html"
    
    # Convert URL to a template path
    path_parts = clean_url.split('/')
    
    # For simple paths, just add .html
    if len(path_parts) == 1:
        return f"{path_parts[0]}.html"
    
    # For nested paths, create appropriate directory structure
    # Last part becomes the filename, rest becomes directories
    filename = path_parts[-1] + ".html"
    directories = path_parts[:-1]
    
    return os.path.join(*directories, filename)

def check_missing_pages():
    """
    Check for missing pages in the templates directory based on sidebar links.
    
    Returns:
        tuple: Lists of (missing_pages, existing_pages)
    """
    missing_pages = []
    existing_pages = []
    
    # Function to process link
    def process_link(link):
        url = link.get('url', '')
        name = link.get('name', 'Unnamed Link')
        
        # Skip external links (those starting with http:// or https://)
        if url.startswith(('http://', 'https://')):
            logger.info(f"Skipping external link: {url}")
            existing_pages.append({
                'name': name,
                'url': url,
                'template_path': None,
                'status': 'external_link'
            })
            return
        
        # Get corresponding template path
        template_path = extract_template_path(url)
        full_path = TEMPLATES_DIR / template_path
        
        if full_path.exists():
            logger.info(f"Template exists: {template_path} for {url}")
            existing_pages.append({
                'name': name,
                'url': url,
                'template_path': template_path,
                'status': 'exists'
            })
        else:
            logger.warning(f"Template missing: {template_path} for {url}")
            missing_pages.append({
                'name': name,
                'url': url,
                'template_path': template_path
            })
    
    # Process all links in the sidebar
    for section in sidebar_config.get('sidebar_sections', []):
        for link in section.get('links', []):
            process_link(link)
            
            # Check for nested links (if any)
            for sublink in link.get('sublinks', []):
                process_link(sublink)
    
    return missing_pages, existing_pages

# Check for missing pages
missing_pages, existing_pages = check_missing_pages()

print(f"Found {len(missing_pages)} missing pages and {len(existing_pages)} existing pages.")

## Create Missing Pages

For each missing page, create a new file in the templates directory with a basic template structure.

In [ ]:
def create_missing_pages(missing_pages):
    """
    Create templates for missing pages.
    
    Args:
        missing_pages (list): List of dictionaries containing missing page information
        
    Returns:
        list: List of dictionaries with creation status
    """
    created_pages = []
    
    for page in missing_pages:
        name = page['name']
        template_path = page['template_path']
        full_path = TEMPLATES_DIR / template_path
        
        try:
            # Create directory if needed
            full_path.parent.mkdir(parents=True, exist_ok=True)
            
            # Render the template
            content = PAGE_TEMPLATE.replace("{{title}}", name)
            
            # Write the file
            with open(full_path, 'w') as f:
                f.write(content)
                
            logger.info(f"Created template: {template_path}")
            
            # Add to created pages list
            created_pages.append({
                'name': name,
                'template_path': template_path,
                'status': 'created',
                'timestamp': datetime.datetime.now().isoformat()
            })
            
        except Exception as e:
            logger.error(f"Error creating template {template_path}: {str(e)}")
            created_pages.append({
                'name': name,
                'template_path': template_path,
                'status': 'error',
                'error': str(e),
                'timestamp': datetime.datetime.now().isoformat()
            })
    
    return created_pages

# Create missing pages
created_pages = create_missing_pages(missing_pages)

print(f"Created {len(created_pages)} missing pages.")

## Update Sidebar Links

Ensure that all sidebar links point to the correct pages in the templates directory.

In [ ]:
def update_sidebar_links():
    """
    Update sidebar links configuration with template information
    
    Returns:
        dict: Updated sidebar configuration
    """
    updated_config = sidebar_config.copy()
    
    # Function to update link with template info
    def update_link_info(link):
        url = link.get('url', '')
        
        # Skip external links
        if url.startswith(('http://', 'https://')):
            return link
        
        # Get corresponding template path
        template_path = extract_template_path(url)
        full_path = TEMPLATES_DIR / template_path
        
        # Add template info to link
        updated_link = link.copy()
        updated_link['template_path'] = str(template_path)
        updated_link['template_exists'] = full_path.exists()
        
        return updated_link
    
    # Update all links in the sidebar
    for i, section in enumerate(updated_config.get('sidebar_sections', [])):
        updated_links = []
        for link in section.get('links', []):
            updated_link = update_link_info(link)
            
            # Update sublinks if any
            if 'sublinks' in link:
                updated_link['sublinks'] = [update_link_info(sublink) for sublink in link['sublinks']]
                
            updated_links.append(updated_link)
            
        updated_config['sidebar_sections'][i]['links'] = updated_links
    
    # Save updated configuration
    updated_config_path = CONFIG_DIR / 'updated_sidebar_config.json'
    with open(updated_config_path, 'w') as file:
        json.dump(updated_config, file, indent=2)
    
    logger.info(f"Saved updated sidebar configuration to {updated_config_path}")
    
    return updated_config

# Update sidebar links
updated_config = update_sidebar_links()

## Generate Status Report

Maintain a running list of links that are complete and working, as well as links that were missing and created.

In [ ]:
def generate_status_report():
    """
    Generate a status report of all sidebar links
    
    Returns:
        dict: Report data
    """
    report = {
        'timestamp': datetime.datetime.now().isoformat(),
        'summary': {
            'total_links': len(existing_pages) + len(missing_pages),
            'existing_pages': len(existing_pages),
            'missing_pages': len(missing_pages),
            'created_pages': len(created_pages),
            'failed_creations': len([p for p in created_pages if p['status'] == 'error'])
        },
        'existing_pages': existing_pages,
        'created_pages': created_pages,
        'failed_pages': [p for p in created_pages if p['status'] == 'error']
    }
    
    # Save the report
    report_path = BASE_DIR / 'reports'
    report_path.mkdir(exist_ok=True)
    
    # Create a timestamped report file
    timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    report_file = report_path / f'sidebar_link_report_{timestamp}.json'
    
    with open(report_file, 'w') as f:
        json.dump(report, f, indent=2)
    
    logger.info(f"Generated status report at {report_file}")
    
    # Also save a current report
    current_report_file = report_path / 'sidebar_link_status.json'
    with open(current_report_file, 'w') as f:
        json.dump(report, f, indent=2)
    
    return report

# Generate status report
report = generate_status_report()

# Display summary
print("Status Report Summary:")
print(f"Total links: {report['summary']['total_links']}")
print(f"Existing pages: {report['summary']['existing_pages']}")
print(f"Missing pages: {report['summary']['missing_pages']}")
print(f"Created pages: {report['summary']['created_pages']}")
print(f"Failed creations: {report['summary']['failed_creations']}")

## Generate HTML Report

Create an HTML report that visually displays the status of all sidebar links.

In [ ]:
def generate_html_report(report):
    """
    Generate an HTML report from the status data
    
    Args:
        report (dict): Report data
        
    Returns:
        str: Path to the HTML report file
    """
    html_content = f"""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Sidebar Link Status Report</title>
    <style>
        body {{ font-family: Arial, sans-serif; margin: 20px; }}
        .summary {{ background-color: #f0f0f0; padding: 15px; border-radius: 5px; margin-bottom: 20px; }}
        table {{ border-collapse: collapse; width: 100%; }}
        th, td {{ border: 1px solid #ddd; padding: 8px; text-align: left; }}
        th {{ background-color: #f2f2f2; }}
        tr:nth-child(even) {{ background-color: #f9f9f9; }}
        .success {{ color: green; }}
        .warning {{ color: orange; }}
        .error {{ color: red; }}
        .external {{ color: blue; }}
    </style>
</head>
<body>
    <h1>Sidebar Link Status Report</h1>
    <p>Report generated on: {datetime.datetime.fromisoformat(report['timestamp']).strftime('%Y-%m-%d %H:%M:%S')}</p>
    
    <div class="summary">
        <h2>Summary</h2>
        <p>Total links: <strong>{report['summary']['total_links']}</strong></p>
        <p>Existing pages: <strong class="success">{report['summary']['existing_pages']}</strong></p>
        <p>Missing pages: <strong class="warning">{report['summary']['missing_pages']}</strong></p>
        <p>Created pages: <strong class="success">{report['summary']['created_pages']}</strong></p>
        <p>Failed creations: <strong class="error">{report['summary']['failed_creations']}</strong></p>
    </div>
    
    <h2>Existing Pages</h2>
    <table>
        <tr>
            <th>Name</th>
            <th>URL</th>
            <th>Template Path</th>
            <th>Status</th>
        </tr>
    """
    
    for page in report['existing_pages']:
        status_class = "external" if page['status'] == 'external_link' else "success"
        template_path = page['template_path'] if page['template_path'] else "N/A (External Link)"
        html_content += f"""
        <tr>
            <td>{page['name']}</td>
            <td>{page['url']}</td>
            <td>{template_path}</td>
            <td class="{status_class}">{page['status']}</td>
        </tr>
        """
    
    html_content += """
    </table>
    
    <h2>Created Pages</h2>
    <table>
        <tr>
            <th>Name</th>
            <th>Template Path</th>
            <th>Status</th>
            <th>Timestamp</th>
        </tr>
    """
    
    for page in report['created_pages']:
        status_class = "success" if page['status'] == 'created' else "error"
        html_content += f"""
        <tr>
            <td>{page['name']}</td>
            <td>{page['template_path']}</td>
            <td class="{status_class}">{page['status']}</td>
            <td>{datetime.datetime.fromisoformat(page['timestamp']).strftime('%Y-%m-%d %H:%M:%S')}</td>
        </tr>
        """
    
    html_content += """
    </table>
    
    <h2>Failed Pages</h2>
    <table>
        <tr>
            <th>Name</th>
            <th>Template Path</th>
            <th>Error</th>
        </tr>
    """
    
    if report['failed_pages']:
        for page in report['failed_pages']:
            html_content += f"""
            <tr>
                <td>{page['name']}</td>
                <td>{page['template_path']}</td>
                <td class="error">{page.get('error', 'Unknown error')}</td>
            </tr>
            """
    else:
        html_content += """
        <tr>
            <td colspan="3" style="text-align: center;">No failed pages</td>
        </tr>
        """
    
    html_content += """
    </table>
</body>
</html>
    """
    
    # Save the HTML report
    report_path = BASE_DIR / 'reports'
    report_path.mkdir(exist_ok=True)
    
    html_report_file = report_path / 'sidebar_link_report.html'
    with open(html_report_file, 'w') as f:
        f.write(html_content)
    
    logger.info(f"Generated HTML report at {html_report_file}")
    
    return str(html_report_file)

# Generate HTML report
html_report_path = generate_html_report(report)
print(f"HTML report generated at: {html_report_path}")

## Conclusion

This notebook has:

1. Loaded the sidebar configuration from a JSON file (or created a sample one)
2. Checked for missing template pages in the `/templates` directory
3. Created templates for any missing pages
4. Updated the sidebar link configuration with template information
5. Generated both a JSON and HTML report of the sidebar link status

The HTML report provides a visual representation of the current status of all sidebar links and can be used to monitor progress as you develop your application.

To run this notebook again:
- When you add new links to your sidebar
- When you reorganize your template structure
- To generate an updated status report